<a href="https://colab.research.google.com/github/Venu-max/NASSCOM-AI/blob/main/Day5_U12_%E2%80%94_Building_ML_Ready_Datasets_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === SETUP: load the provided file (regenerate it if missing) ===
import os
import numpy as np
import pandas as pd


def build_loans(csv_path="loan_applications.csv", seed=23, verbose=False):
    """Realistic loan / credit-risk dataset for building an ML-ready pipeline.

    Built-in realism:
      - imbalanced target (default ~ 16%)
      - mixed numeric + categorical features
      - missing values, a few duplicate rows
      - a DELIBERATELY LEAKY column ('collection_calls') that is only known
        AFTER an account defaults — students must detect & drop it.
    """
    rng = np.random.default_rng(seed)
    N = 4000

    age = np.clip(rng.normal(40, 12, N), 21, 75).round().astype(int)
    income = np.clip(rng.lognormal(11.0, 0.45, N), 12000, None).round(-2)         # right-skewed
    employment_years = np.clip(rng.gamma(3, 2.2, N), 0, 40).round(1)
    credit_score = np.clip(rng.normal(680, 70, N), 300, 850).round().astype(int)
    loan_amount = np.clip(rng.lognormal(10.2, 0.5, N), 1000, None).round(-2)
    loan_term = rng.choice([12, 24, 36, 48, 60], N, p=[.12, .23, .33, .17, .15])
    num_existing_loans = rng.poisson(1.1, N)
    dti = np.clip(rng.normal(25, 10, N) + (loan_amount / (income + 1)) * 15, 2, 90).round(1)
    interest_rate = np.clip(14 - (credit_score - 680) / 35 + rng.normal(0, 1.2, N), 4, 28).round(2)
    home = rng.choice(["Rent", "Own", "Mortgage"], N, p=[.45, .20, .35])
    purpose = rng.choice(["Car", "Home", "Education", "Business", "Personal"],
                         N, p=[.22, .18, .15, .15, .30])
    region = rng.choice(["North", "South", "East", "West", "Central"],
                        N, p=[.24, .22, .18, .20, .16])
    prior_default = rng.choice(["Yes", "No"], N, p=[.14, .86])

    # ---- default risk (real signal) ----
    z = (-2.0
         - 0.012 * (credit_score - 680)
         + 0.035 * (dti - 28)
         + 0.06 * (interest_rate - 12)
         - 0.0000035 * (income - 60000)
         + 0.9 * (prior_default == "Yes")
         + 0.12 * num_existing_loans)
    p = 1 / (1 + np.exp(-z))
    default = (rng.random(N) < p).astype(int)

    # ---- LEAKY feature: collection calls happen only AFTER default ----
    collection_calls = np.where(default == 1, rng.poisson(6, N), rng.poisson(0.2, N))

    df = pd.DataFrame({
        "loan_id": [f"LN{i+1:05d}" for i in range(N)],
        "age": age, "annual_income": income, "employment_years": employment_years,
        "credit_score": credit_score, "loan_amount": loan_amount,
        "loan_term_months": loan_term, "num_existing_loans": num_existing_loans,
        "debt_to_income": dti, "interest_rate": interest_rate,
        "home_ownership": home, "loan_purpose": purpose, "region": region,
        "prior_default": prior_default,
        "collection_calls": collection_calls,            # <-- leakage trap
        "default": default,
    })

    # ---- messiness: missing values + a few duplicates ----
    for col, frac in [("annual_income", 0.05), ("employment_years", 0.06), ("home_ownership", 0.02)]:
        idx = rng.choice(N, int(frac * N), replace=False)
        df.loc[idx, col] = np.nan
    df = pd.concat([df, df.sample(12, random_state=2)], ignore_index=True)

    df.to_csv(csv_path, index=False)
    if verbose:
        print("loans:", df.shape)
        print("default rate:", round(df["default"].mean(), 3))
        corr = df[["collection_calls", "credit_score", "debt_to_income", "default"]].corr()["default"]
        print("corr with default:\n", corr.round(3).to_string())
        print("duplicates:", int(df.duplicated().sum()),
              "| missing income:", int(df["annual_income"].isna().sum()))
    return df

if not os.path.exists('loan_applications.csv'):
    build_loans(); print('Generated dataset file.')
else:
    print('Found the provided dataset file.')

Generated dataset file.


In [2]:
import pandas as pd, numpy as np
df = pd.read_csv('loan_applications.csv')
print('shape:', df.shape)
print('default rate:', round(df['default'].mean(), 3))
df.head(3)

shape: (4012, 16)
default rate: 0.239


,loan_id,age,annual_income,employment_years,credit_score,loan_amount,loan_term_months,num_existing_loans,debt_to_income,interest_rate,home_ownership,loan_purpose,region,prior_default,collection_calls,default
0,LN00001,47,124000.0,8.3,757,18600.0,36,1,19.4,12.33,Own,Car,Central,No,1,0
1,LN00002,43,97200.0,5.0,677,26500.0,12,0,18.9,15.87,Own,Business,East,Yes,0,0
2,LN00003,39,119100.0,5.1,591,16900.0,48,2,37.3,18.65,Rent,Personal,West,No,0,0


In [ ]:
##1. Quick clean (duplicates & a missingness check)

In [3]:
# -----------------------------------------------------------
# 🔹 1A. DROP DUPLICATES; NOTE MISSINGNESS (the pipeline will impute)
# -----------------------------------------------------------
print('duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('after drop:', df.shape)
print('\nmissing values:')
print(df.isna().sum()[lambda s: s > 0])

duplicate rows: 12
after drop: (4000, 16)

missing values:
annual_income       200
employment_years    240
home_ownership       80
dtype: int64


In [ ]:
#2. Separate features (X) and target (y)

In [4]:
# -----------------------------------------------------------
# 🔹 2A. y = what we predict; X = what we're allowed to use
# -----------------------------------------------------------
y = df['default']
X = df.drop(columns=['default', 'loan_id'])   # drop the target and the ID
print('X:', X.shape, '| y:', y.shape)
print('feature columns:', list(X.columns))

X: (4000, 14) | y: (4000,)
feature columns: ['age', 'annual_income', 'employment_years', 'credit_score', 'loan_amount', 'loan_term_months', 'num_existing_loans', 'debt_to_income', 'interest_rate', 'home_ownership', 'loan_purpose', 'region', 'prior_default', 'collection_calls']


In [ ]:
#3. 🔎 Leakage hunt — the cardinal sin

In [5]:

# -----------------------------------------------------------
# 🔹 3A. CORRELATION OF EACH NUMERIC FEATURE WITH THE TARGET
# -----------------------------------------------------------
num_cols = X.select_dtypes('number').columns
corr_y = X[num_cols].corrwith(y).abs().sort_values(ascending=False)
print('Absolute correlation with default:')
print(corr_y.round(3))
print('\nThat top value is suspiciously high — investigate it.')


Absolute correlation with default:
collection_calls      0.893
credit_score          0.321
interest_rate         0.295
debt_to_income        0.170
annual_income         0.094
num_existing_loans    0.083
loan_term_months      0.037
loan_amount           0.033
age                   0.013
employment_years      0.006
dtype: float64

That top value is suspiciously high — investigate it.


In [6]:
# -----------------------------------------------------------
# 🔹 3B. WHY 'collection_calls' IS LEAKAGE
# -----------------------------------------------------------
print(df.groupby('default')['collection_calls'].mean().round(2))
print('\nCollection calls only happen AFTER a loan starts defaulting —')
print("we would NOT know this at application time. It's leakage. Drop it.")
X = X.drop(columns=['collection_calls'])
print('features now:', X.shape[1])


default
0    0.2
1    6.1
Name: collection_calls, dtype: float64

Collection calls only happen AFTER a loan starts defaulting —
we would NOT know this at application time. It's leakage. Drop it.
features now: 13


In [ ]:
#EXERCISE 3 — Prove how badly leakage inflates scores
#This is the most important exercise in the lab.
#Build a quick numeric-only logistic-regression pipeline (impute + scale + model).
#Get its 5-fold CV accuracy using X_leaky = df.drop(columns=['default','loan_id']).select_dtypes('number') (which still contains collection_calls).
#Get the CV accuracy on X.select_dtypes('number') (leakage removed).
#In a comment, report both numbers and explain the gap.

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Create a pipeline:
# 1. Fill missing values with the median
# 2. Standardize the features
# 3. Train a Logistic Regression model

pipe_q = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

# -----------------------------------------------------------
# 1 & 2. CV accuracy WITH the leaky column
# -----------------------------------------------------------

# Create a feature set that still contains 'collection_calls'
X_leaky = df.drop(columns=['default', 'loan_id']).select_dtypes('number')

# Calculate 5-fold cross-validation accuracy
score_leaky = cross_val_score(pipe_q, X_leaky, y, cv=5).mean()

print("Accuracy WITH leakage :", round(score_leaky, 3))


# -----------------------------------------------------------
# 3. CV accuracy WITHOUT the leaky column
# -----------------------------------------------------------

# X already has 'collection_calls' removed
X_clean = X.select_dtypes('number')

# Calculate 5-fold cross-validation accuracy
score_clean = cross_val_score(pipe_q, X_clean, y, cv=5).mean()

print("Accuracy WITHOUT leakage :", round(score_clean, 3))


# -----------------------------------------------------------
# 4. Explanation
# -----------------------------------------------------------

# The model with the leakage feature usually achieves an
# unrealistically high accuracy because 'collection_calls'
# contains information that is only available after a loan
# starts defaulting.
#
# After removing the leakage feature, the accuracy becomes
# more realistic because the model now relies only on
# information available at loan application time.

Accuracy WITH leakage : 0.988
Accuracy WITHOUT leakage : 0.774


In [ ]:
#4. Stratified train/test split

In [8]:
# -----------------------------------------------------------
# 🔹 4A. SPLIT FIRST — and stratify because the target is imbalanced
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('train:', X_train.shape, '| test:', X_test.shape)
print('default rate  train / test:',
      round(y_train.mean(), 3), '/', round(y_test.mean(), 3),
      ' <- preserved by stratify')


train: (3200, 13) | test: (800, 13)
default rate  train / test: 0.239 / 0.239  <- preserved by stratify


In [ ]:
#EXERCISE 4 — Why stratify?
#Make a non-stratified split (same test_size, random_state) and print the train/test default rates.
#Compare with the stratified rates above.
#In a comment, say why stratifying matters more as a class gets rarer.

In [9]:
# -----------------------------------------------------------
# EXERCISE 4 — Why stratify?
# -----------------------------------------------------------

from sklearn.model_selection import train_test_split

# 1. Non-stratified train/test split

X_train_ns, X_test_ns, y_train_ns, y_test_ns = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42   # Same random state, but no stratify
)

# Print default rates

print("Train default rate (Non-stratified):",
      round(y_train_ns.mean(), 3))

print("Test default rate (Non-stratified):",
      round(y_test_ns.mean(), 3))


# 2-3. Compare and explain

# Compare these rates with the stratified split.
# The stratified split preserves nearly the same default
# rate in both training and testing datasets, while the
# non-stratified split may produce different proportions.
#
# Stratification becomes more important when the positive
# class is rare because random sampling may place too few
# (or too many) positive examples into the training or test
# set, leading to unreliable model training and evaluation.

Train default rate (Non-stratified): 0.242
Test default rate (Non-stratified): 0.229


In [ ]:
#5. Handling class imbalance

In [10]:
# -----------------------------------------------------------
# 🔹 5A. WHY ACCURACY LIES ON IMBALANCED DATA
# -----------------------------------------------------------
majority = 1 - y.mean()
print(f'Always predicting "no default" scores {majority:.1%} accuracy —')
print('yet catches ZERO defaulters. Accuracy alone is misleading here.')
print('Fixes: stratify (done), class_weight="balanced", or resampling.')

Always predicting "no default" scores 76.1% accuracy —
yet catches ZERO defaulters. Accuracy alone is misleading here.
Fixes: stratify (done), class_weight="balanced", or resampling.


In [ ]:
#EXERCISE 5 — Balanced vs default model
#Cross-validate a logistic regression with class_weight='balanced' and score with scoring='recall'.
#Do the same without class weights.
#In a comment, say which catches more defaulters (higher recall).

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Separate numerical and categorical columns
num = X.select_dtypes('number').columns.tolist()
cat = X.select_dtypes('object').columns.tolist()

# Preprocessing pipeline
pre = ColumnTransformer([
    (
        'num',
        make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler()
        ),
        num
    ),
    (
        'cat',
        make_pipeline(
            SimpleImputer(strategy='most_frequent'),
            OneHotEncoder(handle_unknown='ignore')
        ),
        cat
    )
])

# -----------------------------------------------------------
# 1. Balanced Logistic Regression Model
# -----------------------------------------------------------

balanced_model = make_pipeline(
    pre,
    LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    )
)

balanced_recall = cross_val_score(
    balanced_model,
    X,
    y,
    cv=5,
    scoring='recall'
).mean()

print("Balanced Model Recall :", round(balanced_recall, 3))


# -----------------------------------------------------------
# 2. Default (Unweighted) Logistic Regression Model
# -----------------------------------------------------------

default_model = make_pipeline(
    pre,
    LogisticRegression(
        max_iter=1000
    )
)

default_recall = cross_val_score(
    default_model,
    X,
    y,
    cv=5,
    scoring='recall'
).mean()

print("Default Model Recall :", round(default_recall, 3))


# -----------------------------------------------------------
# 3. Comment
# -----------------------------------------------------------

# The model with class_weight='balanced' usually achieves
# higher recall because it gives more importance to the
# minority (default) class, allowing it to identify more
# defaulters than the default unweighted model.

Balanced Model Recall : 0.702
Default Model Recall : 0.249


In [ ]:
#6. The leak-free preprocessing pipeline

In [12]:
# -----------------------------------------------------------
# 🔹 6A. ColumnTransformer + model, fitted on TRAIN ONLY
# -----------------------------------------------------------
from sklearn.pipeline import Pipeline
clf = Pipeline([('prep', pre),
                ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])
clf.fit(X_train, y_train)              # every transformer learns from train only
print('test accuracy:', round(clf.score(X_test, y_test), 3))
from sklearn.metrics import recall_score
print('test recall (default class):',
      round(recall_score(y_test, clf.predict(X_test)), 3))

test accuracy: 0.71
test recall (default class): 0.654


In [ ]:
#7. Cross-validation for a stable estimate

In [13]:
# -----------------------------------------------------------
# 🔹 7A. STRATIFIED 5-FOLD CV (mean ± spread)
# -----------------------------------------------------------
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, X, y, cv=5, scoring='roc_auc')
print('ROC-AUC per fold:', scores.round(3))
print(f'mean {scores.mean():.3f}  ±  {scores.std():.3f}')


ROC-AUC per fold: [0.795 0.792 0.758 0.75  0.703]
mean 0.760  ±  0.034


In [ ]:
#EXERCISE 7 — Compare to the leaky version
#Re-run 5-fold ROC-AUC on a pipeline that still includes collection_calls.
#In a comment, contrast it with the clean AUC above — leakage makes the score look too good.

In [14]:
# -----------------------------------------------------------
# EXERCISE 7 — Compare to the leaky version
# -----------------------------------------------------------

# 1. Create a dataset that still contains the leakage column

X_leaky = df.drop(columns=['default', 'loan_id'])

# Separate numeric and categorical columns
num_leaky = X_leaky.select_dtypes('number').columns.tolist()
cat_leaky = X_leaky.select_dtypes('object').columns.tolist()

# Preprocessing pipeline
pre_leaky = ColumnTransformer([
    (
        'num',
        make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler()
        ),
        num_leaky
    ),
    (
        'cat',
        make_pipeline(
            SimpleImputer(strategy='most_frequent'),
            OneHotEncoder(handle_unknown='ignore')
        ),
        cat_leaky
    )
])

# Build the pipeline

clf_leaky = Pipeline([
    ('prep', pre_leaky),
    ('model', LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ))
])

# 5-fold ROC-AUC

scores_leaky = cross_val_score(
    clf_leaky,
    X_leaky,
    y,
    cv=5,
    scoring='roc_auc'
)

print("Leaky ROC-AUC per fold:", scores_leaky.round(3))
print(f"Mean Leaky ROC-AUC: {scores_leaky.mean():.3f}")

Leaky ROC-AUC per fold: [0.995 0.999 0.999 0.991 0.998]
Mean Leaky ROC-AUC: 0.996


In [ ]:
#8. Final ML-readiness check

In [15]:
# -----------------------------------------------------------
# 🔹 8A. GATE CHECKS — assert the dataset is truly ready
# -----------------------------------------------------------
checks = {
    'no leaky column': 'collection_calls' not in X.columns,
    'X and y aligned': len(X) == len(y),
    'target is binary': set(y.unique()) == {0, 1},
    'split is stratified': abs(y_train.mean() - y_test.mean()) < 0.02,
    'reproducible seed used': True,
}
for k, v in checks.items():
    print(('PASS' if v else 'FAIL'), '-', k)
print('\nReady for modelling:', all(checks.values()))

PASS - no leaky column
PASS - X and y aligned
PASS - target is binary
PASS - split is stratified
PASS - reproducible seed used

Ready for modelling: True


In [ ]:
#EXERCISE 8 — Add your own gate
#Add a check that no feature column is missing after the pipeline transforms the training data (hint: clf.named_steps['prep'].transform(X_train) then check for NaNs with np.isnan(...).any()).
#Add a check that there are no duplicate loan_ids in the original df.
#Print PASS/FAIL for both.

In [16]:
# -----------------------------------------------------------
# EXERCISE 8 — Add your own gate
# -----------------------------------------------------------

import numpy as np

# -----------------------------------------------------------
# 1. Check that there are no NaN values after preprocessing
# -----------------------------------------------------------

# Transform the training data using the fitted preprocessing pipeline
X_train_transformed = clf.named_steps['prep'].transform(X_train)

# Check whether any NaN values exist
no_nan_after_preprocessing = not np.isnan(X_train_transformed).any()


# -----------------------------------------------------------
# 2. Check that loan_id values are unique
# -----------------------------------------------------------

# duplicated().sum() returns the number of duplicate loan_ids
unique_loan_id = (df['loan_id'].duplicated().sum() == 0)


# -----------------------------------------------------------
# 3. Print PASS/FAIL results
# -----------------------------------------------------------

print(("PASS" if no_nan_after_preprocessing else "FAIL"),
      "- No NaN values after preprocessing")

print(("PASS" if unique_loan_id else "FAIL"),
      "- loan_id values are unique")

PASS - No NaN values after preprocessing
PASS - loan_id values are unique
